# Building AI Agent

## Goal

This notebook is a hands-on journey to build an AI agent from scratch.

Each version introduces one new concept, allowing the agent to evolve step by step while practicing AI agent development.

---

## Version 5

In this version, we introduce a simple Memory Manager.

The agent no longer writes directly to memory using `memory.append(state)`.
Instead, it delegates memory operations to dedicated functions.

The memory is still a simple list, but it is now managed through a specific component.

The main goal of this version is to separate memory management from the agent coordination logic.

## 1. Imports

In [1]:
import re

## 2. Tools

In [2]:
# Tools

def greeting(name):
    """Greet the given name."""
    return f"Hello {name.title()}, nice to meet you!"


def addition(a, b):
    """Add two numbers."""
    return a + b


def subtraction(a, b):
    """Subtract two numbers."""
    return a - b


def multiplication(a, b):
    """Multiply two numbers."""
    return a * b


def division(a, b):
    """Divide two numbers."""
    if b == 0:
        return "Error: division by zero!"
    return a / b


def power(a, b):
    """Raise a number to a power."""
    return a ** b

In [3]:
# Tool metadata

tools = {
    "greeting": {
        "function": greeting,
        "parameters": ["name"],
        "description": "Greet the user by name."
    },
    "addition": {
        "function": addition,
        "parameters": ["a", "b"],
        "description": "Add two numbers."
    },
    "subtraction": {
        "function": subtraction,
        "parameters": ["a", "b"],
        "description": "Subtract two numbers."
    },
    "multiplication": {
        "function": multiplication,
        "parameters": ["a", "b"],
        "description": "Multiply two numbers."
    },
    "division": {
        "function": division,
        "parameters": ["a", "b"],
        "description": "Divide two numbers."
    },
    "power": {
        "function": power,
        "parameters": ["a", "b"],
        "description": "Raise a number to a power."
    }
}

In [4]:
# Tool synonyms

tool_synonyms = {
    "greeting": ["hello", "hi", "hey", "good morning", "good afternoon", "good evening"],
    "addition": ["add", "addition", "sum", "plus", "+"],
    "subtraction": ["subtract", "subtraction", "minus", "take away","-"],
    "multiplication": ["multiply", "multiplication", "times","*"],
    "division": ["divide", "division", "divided by", "/"],
    "power": ["power", "raise", "raised to", "**"]
}

## 3. Parser

In [5]:
# Extract name

def extract_name(text):
    match = re.search(
        r"(?:my name is|i am|i'm|this is)\s+([A-Z][a-z]+)",
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1)

    words = re.findall(r"\b[A-Z][a-z]+\b", text)

    return words[1] if len(words) > 1 else None

In [6]:
# Extract numbers

def extract_numbers(text):
    numbers = []

    for token in text.split():
        token = re.sub(r"[^\w\s]", "", token)

        if token.isdigit():
            numbers.append(int(token))

    return numbers

In [7]:
# Parser

def build_parse_context(request):
    return {
        "numbers": extract_numbers(request),
        "name": extract_name(request)
    }


def extract_first_number(request, context):
    numbers = context["numbers"]
    return numbers[0] if len(numbers) > 0 else None


def extract_second_number(request, context):
    numbers = context["numbers"]
    return numbers[1] if len(numbers) > 1 else None


parameter_extractors = {
    "name": lambda request, context: context["name"],
    "a": extract_first_number,
    "b": extract_second_number
}


def parser(tool_name, request):
    if tool_name is None:
        return {}

    tool_info = tools.get(tool_name)

    if not tool_info:
        return {}

    context = build_parse_context(request)
    arguments = {}

    for parameter in tool_info["parameters"]:
        extractor = parameter_extractors.get(parameter)

        if extractor is None:
            arguments[parameter] = None
        else:
            arguments[parameter] = extractor(request, context)

    return arguments

## 4. Router

In [8]:
# Router

def choose_tool(request):
    request = request.lower()
    scores = {}

    for tool_name, synonyms in tool_synonyms.items():
        score = 0

        for synonym in synonyms:
            if synonym in request:
                score += 1

        if score > 0:
            scores[tool_name] = score

    if not scores:
        return None

    priority = {
        "greeting": 1,
        "addition": 10,
        "subtraction": 10,
        "multiplication": 10,
        "division": 10,
        "power": 10
    }

    best_tool = max(
        scores,
        key=lambda tool: (scores[tool], priority.get(tool, 0))
    )

    return best_tool

## 5. Planner

In [9]:
# Planner

def planner(request):
    tool_name = choose_tool(request)

    if tool_name is None:
        return {
            "tool": None,
            "reason": "No suitable tool was found for the request.",
            "needs_arguments": []
        }

    tool_info = tools.get(tool_name)

    return {
        "tool": tool_name,
        "reason": tool_info["description"],
        "needs_arguments": tool_info["parameters"]
    }

## 6. Executor

In [10]:
# Validate arguments

def arguments_are_valid(arguments):
    if not arguments:
        return False

    for value in arguments.values():
        if value is None:
            return False

    return True

In [11]:
# Executor

def executor(tool_name, arguments):
    tool_info = tools.get(tool_name)

    if not tool_info:
        return "Tool not found!"

    if not arguments_are_valid(arguments):
        return "Invalid arguments"

    function = tool_info["function"]

    return function(**arguments)

## 7. Memory Manager

In [12]:
# Memory Manager

memory = []


def save_to_memory(state):
    memory.append(state)


def get_memory():
    return memory


def get_last_interaction():
    if not memory:
        return None

    return memory[-1]


def clear_memory():
    memory.clear()

## 8. Agent

In [13]:
# Agent

def agent(request):
    state = {
        "request": request,
        "plan": None,
        "action": None,
        "arguments": {},
        "result": None
    }

    plan = planner(request)
    state["plan"] = plan

    action = plan["tool"]
    state["action"] = action

    if action is None:
        state["result"] = "I cannot handle this request yet."
        save_to_memory(state)
        return state

    arguments = parser(action, request)
    state["arguments"] = arguments

    result = executor(action, arguments)
    state["result"] = result

    save_to_memory(state)

    return state

## 9. Tests

In [14]:
clear_memory()

In [15]:
agent("What is 2 + 3?")

{'request': 'What is 2 + 3?',
 'plan': {'tool': 'addition',
  'reason': 'Add two numbers.',
  'needs_arguments': ['a', 'b']},
 'action': 'addition',
 'arguments': {'a': 2, 'b': 3},
 'result': 5}

In [16]:
agent("Hello, what is 2 + 3?")

{'request': 'Hello, what is 2 + 3?',
 'plan': {'tool': 'addition',
  'reason': 'Add two numbers.',
  'needs_arguments': ['a', 'b']},
 'action': 'addition',
 'arguments': {'a': 2, 'b': 3},
 'result': 5}

In [17]:
agent("Hello, my name is luca.")

{'request': 'Hello, my name is luca.',
 'plan': {'tool': 'greeting',
  'reason': 'Greet the user by name.',
  'needs_arguments': ['name']},
 'action': 'greeting',
 'arguments': {'name': 'luca'},
 'result': 'Hello Luca, nice to meet you!'}

In [18]:
agent("What is the weather in Rome?")

{'request': 'What is the weather in Rome?',
 'plan': {'tool': None,
  'reason': 'No suitable tool was found for the request.',
  'needs_arguments': []},
 'action': None,
 'arguments': {},
 'result': 'I cannot handle this request yet.'}

In [19]:
get_memory()

[{'request': 'What is 2 + 3?',
  'plan': {'tool': 'addition',
   'reason': 'Add two numbers.',
   'needs_arguments': ['a', 'b']},
  'action': 'addition',
  'arguments': {'a': 2, 'b': 3},
  'result': 5},
 {'request': 'Hello, what is 2 + 3?',
  'plan': {'tool': 'addition',
   'reason': 'Add two numbers.',
   'needs_arguments': ['a', 'b']},
  'action': 'addition',
  'arguments': {'a': 2, 'b': 3},
  'result': 5},
 {'request': 'Hello, my name is luca.',
  'plan': {'tool': 'greeting',
   'reason': 'Greet the user by name.',
   'needs_arguments': ['name']},
  'action': 'greeting',
  'arguments': {'name': 'luca'},
  'result': 'Hello Luca, nice to meet you!'},
 {'request': 'What is the weather in Rome?',
  'plan': {'tool': None,
   'reason': 'No suitable tool was found for the request.',
   'needs_arguments': []},
  'action': None,
  'arguments': {},
  'result': 'I cannot handle this request yet.'}]

In [20]:
get_last_interaction()

{'request': 'What is the weather in Rome?',
 'plan': {'tool': None,
  'reason': 'No suitable tool was found for the request.',
  'needs_arguments': []},
 'action': None,
 'arguments': {},
 'result': 'I cannot handle this request yet.'}

## Notes

Some test cases were intentionally designed to verify specific parts of the agent.

- `"Hello, what is 2 + 3?"` checks that the router correctly prioritizes the user's main intent (addition) over a greeting.
- `"Hello, my name is luca."` uses a lowercase name on purpose to verify that the tool formats the output (`Luca`) instead of relying on the user's input.
- `"What is the weather in Rome?"` checks that the planner correctly handles unsupported requests by creating a plan with no selected tool, and that the interaction is still saved through the memory manager.
- The router is responsible for selecting the most suitable tool for the user request.
- The parser reads the parameters required by the selected tool and extracts the needed arguments.
- The planner creates a simple execution plan before the agent parses arguments and executes the selected tool.
- The plan contains the selected tool, the reason for using it, and the arguments required by the tool.
- The executor is responsible for validating the parsed arguments and running the selected tool.
- The memory manager is responsible for saving, reading, and clearing memory.
- The agent coordinates the full flow: planning, parsing, execution, and memory storage through the memory manager.
- The memory object stores the state of every interaction, allowing the agent to keep track of previous requests, plans, actions, arguments, and results.

Future versions will introduce new components and gradually evolve the architecture.